### Імпорт усіх необхідних бібліотек

In [1]:
%matplotlib tk 
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button, CheckButtons
from scipy.signal import butter, filtfilt

### Генерування та фліьтрація

In [2]:
def generate_base_harmonic(t, amplitude, frequency, phase):
    return amplitude * np.sin(2 * np.pi * frequency * t + phase)

def generate_noise_array(length, mean, covariance):
    std_dev = np.sqrt(covariance)
    return np.random.normal(mean, std_dev, length)

def harmonic_with_noise(t, amplitude, frequency, phase, noise_mean, noise_covariance, show_noise, current_noise=None):
    y_pure = generate_base_harmonic(t, amplitude, frequency, phase)
    
    if show_noise:
        if current_noise is None:
            current_noise = generate_noise_array(len(t), noise_mean, noise_covariance)
        return y_pure, y_pure + current_noise, current_noise
    else:
        return y_pure, y_pure, current_noise

def apply_lowpass_filter(data, cutoff_freq, fs, order=4):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff_freq / nyquist
    if normal_cutoff >= 1.0 or normal_cutoff <= 0.0:
        return data
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return filtfilt(b, a, data)

### Створення головного вікна з графіком, слайдерами (амплітуда, частота, параметри шуму, фільтрації), кнопкою Reset та чекбоксом відображення шуму. Реалізація логіки оновлення графіків без перегенерації шуму при зміні параметрів гармоніки.

In [3]:
class InteractiveHarmonicApp:
    def __init__(self):
        self.t = np.linspace(0, 10, 1000)
        self.fs = 1000 / 10 
        
        self.init_amp, self.init_freq, self.init_phase = 0.97, 0.267, 0.0
        self.init_mean, self.init_cov = 0.108, 0.101
        self.init_cutoff = 5.0
        
        self.show_noise_flag = True
        self.current_noise = generate_noise_array(len(self.t), self.init_mean, self.init_cov)
        
        self.fig, self.ax = plt.subplots(figsize=(10, 6.5))
        plt.subplots_adjust(bottom=0.45)
        
        y_pure, y_noisy, _ = harmonic_with_noise(
            self.t, self.init_amp, self.init_freq, self.init_phase, 
            self.init_mean, self.init_cov, True, self.current_noise
        )
        y_filtered = apply_lowpass_filter(y_noisy, self.init_cutoff, self.fs)
        
        self.line_pure, = self.ax.plot(self.t, y_pure, 'b--', label='Pure Harmonic', linewidth=2)
        self.line_noisy, = self.ax.plot(self.t, y_noisy, 'orange', alpha=0.7, label='Noisy Signal')
        self.line_filtered, = self.ax.plot(self.t, y_filtered, 'purple', label='Filtered Signal', linewidth=2)
        
        self.ax.set_ylim(-2.5, 2.5)
        self.ax.legend(loc='upper right')
        self.ax.set_title("Гармоніка з накладеним шумом та фільтрацією")
        
        axcolor = 'lightgoldenrodyellow'
        ax_amp = plt.axes([0.25, 0.35, 0.65, 0.03], facecolor=axcolor)
        ax_freq = plt.axes([0.25, 0.30, 0.65, 0.03], facecolor=axcolor)
        ax_phase = plt.axes([0.25, 0.25, 0.65, 0.03], facecolor=axcolor)
        ax_mean = plt.axes([0.25, 0.20, 0.65, 0.03], facecolor=axcolor)
        ax_cov = plt.axes([0.25, 0.15, 0.65, 0.03], facecolor=axcolor)
        ax_cutoff = plt.axes([0.25, 0.10, 0.65, 0.03], facecolor=axcolor)
        
        self.s_amp = Slider(ax_amp, 'Amplitude', 0.1, 2.0, valinit=self.init_amp)
        self.s_freq = Slider(ax_freq, 'Frequency', 0.01, 2.0, valinit=self.init_freq)
        self.s_phase = Slider(ax_phase, 'Phase', 0.0, 2*np.pi, valinit=self.init_phase)
        self.s_mean = Slider(ax_mean, 'Noise Mean', -1.0, 1.0, valinit=self.init_mean)
        self.s_cov = Slider(ax_cov, 'Noise Covariance', 0.0, 1.0, valinit=self.init_cov)
        self.s_cutoff = Slider(ax_cutoff, 'Cutoff Freq', 0.1, 20.0, valinit=self.init_cutoff)
        
        ax_reset = plt.axes([0.05, 0.025, 0.1, 0.04])
        self.btn_reset = Button(ax_reset, 'Reset', color=axcolor)
        
        ax_check = plt.axes([0.8, 0.025, 0.15, 0.05])
        self.check_noise = CheckButtons(ax_check, ['Show Noise'], [self.show_noise_flag])
        
        self.s_amp.on_changed(self.update_plot)
        self.s_freq.on_changed(self.update_plot)
        self.s_phase.on_changed(self.update_plot)
        self.s_mean.on_changed(self.update_noise_params)
        self.s_cov.on_changed(self.update_noise_params)
        self.s_cutoff.on_changed(self.update_plot)
        
        self.btn_reset.on_clicked(self.reset)
        self.check_noise.on_clicked(self.toggle_noise)
        
        plt.show()

    def update_noise_params(self, val):
        self.current_noise = generate_noise_array(len(self.t), self.s_mean.val, self.s_cov.val)
        self.update_plot(val)

    def toggle_noise(self, label):
        self.show_noise_flag = not self.show_noise_flag
        self.line_noisy.set_visible(self.show_noise_flag)
        self.fig.canvas.draw_idle()

    def update_plot(self, val):
        y_pure, y_noisy, _ = harmonic_with_noise(
            self.t, self.s_amp.val, self.s_freq.val, self.s_phase.val, 
            self.s_mean.val, self.s_cov.val, True, self.current_noise
        )
        y_filtered = apply_lowpass_filter(y_noisy, self.s_cutoff.val, self.fs)
        
        self.line_pure.set_ydata(y_pure)
        self.line_noisy.set_ydata(y_noisy)
        self.line_filtered.set_ydata(y_filtered)
        self.fig.canvas.draw_idle()

    def reset(self, event):
        self.s_amp.reset()
        self.s_freq.reset()
        self.s_phase.reset()
        self.s_mean.reset()
        self.s_cov.reset()
        self.s_cutoff.reset()
        if not self.show_noise_flag:
            self.check_noise.set_active(0)
        self.update_plot(None)

app = InteractiveHarmonicApp()

### Результат роботи програми
Оскільки через некоректне відображення браузера було використано `%matplotlib tk`, і вивід тепер працює через окреме вікно, нижче надаю скріншоти відпрацьованої програми:

![Result](result1.png)

![Result](result2.png)